# P&ID Medallion Pipeline — Concepts Walkthrough (Bronze → Silver)

This notebook illustrates, end to end, what we have built so far: a **Bronze**
(raw, immutable ingestion) → **Silver** (parse + topology reconstruction) pipeline
for P&ID interoperability exports (DEXPI/Proteus and INGR ISO-15926 PostProc),
on local Spark + Delta Lake.

It demonstrates the key concepts:

- **Bronze** stores the source XML *as-is* — content hash, format detection, project
  code and drawing revision captured, dedup on exact bytes.
- **Silver** *re-houses* the validated `pidtool`/`bppidsys` reconstruction (the
  "crown jewel") — recovering inline valves the raw graph lacks — and emits typed
  tables: components, segments, connections, equipment.
- The **oracle firewall**: the source turnover assignment is carried as *quarantined*
  lineage, never computed on.
- The **`flow_sense`** four-state directional overlay and the **`derived`** provenance
  flag on every reified connection.
- **Format parity**: DEXPI and PostProc flow through one code path into one schema.
- A real-data finding: **`seg_tag` is not unique** (the CDC anchor-collision risk).
- **Stage D**: a declarative expectation suite writes the **`silver_quality`**
  punch list; only two structural invariants hard-fail (fail for bugs, not data).

> **Run order matters.** After any kernel restart, run the cells top-to-bottom.
> Cell 1 *must* be first — it forces the venv's Spark 3.5.1 and blocks the system
> Spark 4 at `/opt/spark`.

## 0. Environment & pinned session

Two things bite on local WSL and are handled here:

1. **Which Spark.** A system `SPARK_HOME=/opt/spark` (Spark 4) shadows the venv's
   Spark 3.5.1 and breaks Delta (`DeltaCatalog` not found). Cell 1 strips it
   *before* `pyspark` is ever imported.
2. **One catalog, one warehouse.** The Hive metastore (`metastore_db/`) holds
   *names → locations*; the warehouse (`spark-warehouse/`) holds the *data*. We
   **pin both** to fixed paths so every session sees the same tables (embedded
   Derby is single-session — don't also run a `!python -m ...` subprocess while
   this notebook's session is live).

In [ ]:
# --- CELL 1 — must run FIRST (before any `import pyspark`) ---
import os, sys
os.environ.pop("SPARK_HOME", None)                       # ignore system /opt/spark (Spark 4)
os.environ["PYTHONPATH"] = os.pathsep.join(
    p for p in os.environ.get("PYTHONPATH", "").split(os.pathsep) if "/opt/spark" not in p)
sys.path[:] = [p for p in sys.path if "/opt/spark" not in p]
assert "pyspark" not in sys.modules, "Restart the kernel and run THIS cell first."

from pathlib import Path

# repo_root = the ProjectData repo root (the folder containing bronze/ and silver/)
repo_root = Path.cwd()
while not (repo_root / "bronze").is_dir() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
assert (repo_root / "bronze").is_dir(), f"Open this notebook inside the ProjectData repo (cwd={Path.cwd()})"

# --- the variables for this walkthrough ---
source_sample_dir = repo_root / "sample_data"                 # committed synthetic fixtures
source_dir        = repo_root / "data/exports/projectA"       # real Project A (DEXPI)
source_dir_B      = repo_root / "data/exports/projectB"       # real Project B (PostProc)
table_path        = repo_root / "_tmp" / "bronze_pid_documents"  # PATH-BASED Bronze (throwaway)
spark_warehouse   = repo_root / "spark-warehouse"             # managed-table data
metastore_db      = repo_root / "metastore_db"                # Hive/Derby catalog

print("repo_root       :", repo_root)
print("table_path      :", table_path)
print("spark_warehouse :", spark_warehouse)
print("metastore_db    :", metastore_db)

In [ ]:
# --- CELL 2 — build ONE pinned Delta+Hive session (venv Spark 3.5.1) ---
from bronze.spark_session import get_spark
spark = get_spark(extra_conf={
    "spark.sql.warehouse.dir": f"file:{spark_warehouse}",
    "spark.hadoop.javax.jdo.option.ConnectionURL":
        f"jdbc:derby:;databaseName={metastore_db};create=true",
})
import pyspark
from pyspark.sql import functions as F
print("pyspark :", pyspark.__file__)   # expect .../.venv/...  NOT /opt/spark
print("Spark   :", spark.version)      # expect 3.5.1
assert "/opt/spark" not in pyspark.__file__, "Still on system Spark 4 — restart kernel, run Cell 1 first."
assert spark.version.startswith("3.5"), f"Expected Spark 3.5.x, got {spark.version}"
print("OK — venv Spark 3.5.1, Delta + Hive ready.")

In [ ]:
# --- CELL 3 (optional) — clean slate for a reproducible demo ---
# Safe: _tmp Bronze is throwaway; Silver tables are rebuilt from Bronze below.
import shutil
shutil.rmtree(table_path, ignore_errors=True)
spark.sql("CREATE DATABASE IF NOT EXISTS silver")
for t in ["silver_components", "silver_segments", "silver_connections", "silver_equipment"]:
    spark.sql(f"DROP TABLE IF EXISTS silver.{t}")
    shutil.rmtree(spark_warehouse / "silver.db" / t, ignore_errors=True)
print("clean slate ready")

## 1. Bronze — raw, immutable ingestion

Bronze lands each source file **verbatim**, one row per distinct byte-version, with:
`content` (raw bytes), a self-describing `content_hash` (`sha256:…`), the detected
`source_format` (DEXPI vs POSTPROC), the EPC `document_number` and derived
`project_code`, and the current `drawing_revision` / `drawing_revision_date`.
It **never interprets** the network model — that's Silver's job.

Here we ingest into a **path-based** Bronze table (`table_path`), which needs no
metastore at all.

In [ ]:
# --- pick sources: prefer the real exports, fall back to the committed samples ---
def xmls(d): return sorted(Path(d).glob("*.xml")) if Path(d).is_dir() else []
sources = [d for d in (source_dir, source_dir_B) if xmls(d)]
if not sources:
    sources = [source_sample_dir]
for d in sources:
    print(f"{len(xmls(d)):3d} xml  in  {d}")

In [ ]:
# --- ingest each source folder into the SAME path-based Bronze table ---
from bronze.notebook import ingest_folder
for d in sources:
    summary = ingest_folder(spark, source_dir=str(d), table_path=str(table_path))
    print(d.name, "->", summary)

In [ ]:
# --- inspect Bronze: both formats, lineage columns, self-describing hash ---
bronze = spark.read.format("delta").load(str(table_path))
print("Bronze rows:", bronze.count())
bronze.groupBy("source_format").count().show()
bronze.select("document_number", "drawing_revision", "drawing_revision_date",
              "project_code", "content_hash").show(6, False)

**Dedup on exact bytes.** Re-ingesting the same files lands *nothing* new —
Bronze versions files by `content_hash`, so identical bytes are skipped
(`rows_skipped_already_present`).

In [ ]:
# re-ingest the first folder — expect rows_inserted: 0
print(ingest_folder(spark, source_dir=str(sources[0]), table_path=str(table_path)))

## 2. Silver — parse + topology reconstruction

Silver reads Bronze, picks the adapter from `source_format`, builds the DOM from
the raw bytes, and runs the **validated reconstruction** (vendored under
`silver/_recon/`, re-housed not re-derived). It emits four typed Delta tables and
carries the source turnover assignment as **quarantined** lineage.

We run it **in-session** (same notebook session) reading Bronze by path — so the
Silver tables land in this session's pinned catalog and are queryable by name.

In [ ]:
# --- run Silver Stage A+B in-session ---
from silver.notebook import reconstruct
counts = reconstruct(spark, bronze_path=str(table_path), silver_schema="silver")
print(counts)

In [ ]:
for t in ["silver_components", "silver_segments", "silver_connections", "silver_equipment"]:
    print(f"{t:22s} {spark.table('silver.' + t).count():6d} rows")

## 3. The concepts, illustrated in the data

### 3a. The crown jewel — inline valves recovered

The raw `<Connection>` records wire only each segment's two endpoints; inline valves
are missing. The reconstruction repairs the topology and re-inserts them. Here they
appear as real components flagged `is_valve` — in **both** formats.

In [ ]:
spark.table("silver.silver_components") \
     .groupBy("source_format", "is_valve").count() \
     .orderBy("source_format", "is_valve").show()

### 3b. The oracle firewall

`src_turnover` / `src_subsystem` (the source commissioning assignment) is **carried**
on the segment row — but it sits on its own columns and **nothing computes on it**.
It is the validation *answer key*, quarantined so the ~97% agreement stays honest.

In [ ]:
spark.table("silver.silver_segments") \
     .select("seg_tag", "fluid", "piping_materials_class",
             "src_turnover", "src_subsystem", "project_code").show(6, False)

### 3c. `flow_sense` — the four-state directional overlay

Direction is a *separate overlay* on the undirected connection, and it has four
states — `none` and `both` are real and a boolean couldn't hold them. Both formats
produce all four.

In [ ]:
spark.table("silver.silver_connections") \
     .groupBy("source_format", "flow_sense").count() \
     .orderBy("source_format", "flow_sense").show()

### 3d. `derived` — Source (stated) vs Derived (reconstructed) edges

Every reified connection carries provenance: `derived=false` where the edge was
stated in a source `<Connection>`, `derived=true` where the reconstruction inferred
it. This is what keeps the semantic layer from asserting inferred topology as fact.

In [ ]:
spark.table("silver.silver_connections").groupBy("source_format", "derived").count().show()

### 3e. Format parity — two standards, one schema

DEXPI and PostProc coexist in the same tables with identical columns — the
interoperability promise made concrete.

In [ ]:
spark.table("silver.silver_segments").groupBy("source_format").count().show()

### 3f. Real-data finding — `seg_tag` is not unique

Distinct `segment_id`s can compose to the **same** business `seg_tag`. So the
composed tag cannot stand alone as the CDC segment anchor — it needs a
disambiguator, and the quality gate owes an *anchor-collision* flag. (This is why
we recorded it in the spec's §3.5.)

In [ ]:
(spark.table("silver.silver_segments")
   .groupBy("seg_tag").count().filter("count > 1")
   .orderBy(F.desc("count")).show(10, False))

## 3g. Stage D — the data-quality punch list

Stage D promotes the specs' advisory flags to a **declarative expectation suite**
(rules-as-data, `silver/quality_suite.py`) and writes `silver_quality` — the
per-drawing / per-project **punch list** a pre-commissioning engineer fixes at
source *before* systemization runs (segments missing fluid / piping-class /
diameter, tags that break the naming convention, the `seg_tag` anchor-collision,
prefix-integrity, orphans). The gate is **observe-and-record**: everything flags
and flows. Only two *structural invariants* — an **oracle leak** (§5) or an
**unflagged `derived` edge** (§4) — hard-fail, because those are pipeline bugs,
not dirty data.

It also denormalises a `quality_gate` enum (`clean`/`flagged`/`quarantined`) back
onto every object row, so a cautious consumer can filter without joining the
ledger.

In [ ]:
# --- run Stage D in-session; it reads the four Silver tables ---
from silver.notebook import quality
# refdata_path lights up the reference-backed checks (unknown fluid/unit, naming);
# without it those skip cleanly. Point it at the project's Reference_Data.xlsx:
refdata_path = repo_root / "Reference_Data.xlsx"
summary = quality(spark, refdata_path=str(refdata_path) if refdata_path.exists() else None)
import json; print(json.dumps(summary, indent=2, default=str))

**The punch list** — one row per flag occurrence, ordered worst-first. This
is the artefact the engineer works from.

In [ ]:
from pyspark.sql import functions as F
sev_rank = F.when(F.col("severity") == "error", 0).when(F.col("severity") == "warn", 1).otherwise(2)
(spark.table("silver.silver_quality")
   .withColumn("_r", sev_rank)
   .orderBy("_r", "flag")
   .select("severity", "gate", "object_kind", "flag", "drawing_number", "detail")
   .show(40, False))

**Punch-list rollup by flag** — where the data gaps concentrate.

In [ ]:
(spark.table("silver.silver_quality")
   .groupBy("flag", "severity", "gate").count()
   .orderBy(F.desc("count")).show(30, False))

**The gate rollup on the objects themselves** — `flagged` rows still flow to
Gold and the rules; a strict consumer can exclude `quarantined` without a join.

In [ ]:
for t in ["silver_segments", "silver_components"]:
    print(t)
    spark.table("silver." + t).groupBy("quality_gate").count().orderBy("quality_gate").show()

## 3h. Lineage trace — one attribute, Bronze bytes → Silver column

The whole point of carrying `bronze_id` / `content_hash` on every Silver row (§4)
is that any value traces back to the exact source bytes it came from. Here we
follow **insulation** end to end: the Silver `insul_purpose` column, the raw
`InsulPurpose` attribute pulled straight out of the Bronze XML, and the `seg_tag`
suffix (e.g. `-H`) are the *same source fact reached three ways*. Reading it from
the segment's own `<GenericAttributes>` block mirrors `pidsys.master_data.ga()`
exactly. The identical three-hop walk traces fluid, diameter, piping class, or the
quarantined oracle columns — insulation isn't special.

In [ ]:
# --- Lineage trace: Insulation from Bronze bytes -> Silver columns ---
import xml.etree.ElementTree as ET

seg = spark.table("silver.silver_segments")

# 1) the Silver insulation columns + the lineage keys that trace each row to source
(seg.select("segment_id", "seg_tag", "insul_purpose", "insul_type", "insul_thick",
            "bronze_id", "content_hash", "drawing_number")
    .where("insul_purpose is not null")
    .show(8, False))

# 2) pull InsulPurpose straight out of the raw Bronze XML for one segment and compare.
#    Bronze is a named table in a full run; fall back to the path-based table (Cell 1).
try:
    bronze = spark.table("bronze.pid_documents")
except Exception:
    bronze = spark.read.format("delta").load(str(table_path))

row = (seg.where("insul_purpose is not null")
          .join(bronze.select("bronze_id", "content"), "bronze_id")
          .select("segment_id", "insul_purpose", "insul_type", "insul_thick", "content")
          .head())

def _ln(el):                                   # strip XML namespace
    return el.tag.split("}")[-1]

def insul_from_bytes(content, seg_id):
    # mirror pidsys.master_data.ga(): the segment's OWN <GenericAttributes> block
    root = ET.fromstring(bytes(content))
    for el in root.iter():
        if _ln(el) == "PipingNetworkSegment" and el.get("ID") == seg_id:
            return {g.get("Name"): g.get("Value")
                    for gas in el if _ln(gas) == "GenericAttributes"
                    for g in gas if _ln(g) == "GenericAttribute"
                    and (g.get("Name") or "").startswith("Insul")}
    return {}

if row is None:
    print("no segment with a non-null insul_purpose yet — run Silver Stage A+B first")
else:
    tag = seg.where(seg.segment_id == row.segment_id).head().seg_tag
    print("segment_id :", row.segment_id)
    print("seg_tag    :", tag, "  (last token = insulation purpose)")
    print("SILVER cols:", dict(insul_purpose=row.insul_purpose,
                               insul_type=row.insul_type, insul_thick=row.insul_thick))
    print("BRONZE XML :", insul_from_bytes(row.content, row.segment_id))
    # to trace a SPECIFIC flagged segment: replace the filter in `row` with
    #   .where("segment_id = '<the id from silver_quality.object_id>'")

> If `insul_purpose` comes back all-null in Silver while the `seg_tag` still
> shows a `-H`/`-N` suffix, that mismatch *is* the finding — the value reached the
> composed tag but the column extraction missed it (Bronze→Silver drift), which is
> exactly what this trace is built to catch.

## 4. Recap

**Built (Phase-1 + Stage D):** Bronze (raw, immutable, dedup, format-tagged) →
Silver (parse + reconstruction, four typed tables) → **Stage D quality gate**
(`silver_quality` punch list + `quality_gate` rollup), validated on real Project A
**and** Project B.

**Concepts shown:** store-as-is + content hash; format detection; the reconstruction
recovering inline valves; the oracle firewall; the `flow_sense` enum and `derived`
provenance; format parity; the `seg_tag` anchor-collision; and the Stage-D
punch list with its fail-for-bugs-not-data gate policy.

**Runtime lessons baked in:** force the venv's Spark 3.5.1 (Cell 1); pin the metastore
+ warehouse; run in-session; Bronze can be path-based to sidestep the catalog entirely.

**Next:** Stage C (OPC cross-document assembly) and Stage E (object-grain CDC),
then the Gold layer (bi-temporal + RDF/IDO). Stage D's reference-backed checks
(unknown fluid/unit, equipment/instrument naming) light up as soon as a project
`Reference_Data.xlsx` — with a `Naming` sheet — is supplied.